In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
import matplotlib.pyplot as plt

from src.constants import (
    CA_NAD83_Albers,
    CULVER_CITY_FEED_KEY,
    SERVICE_DATE,
    SHAPE_KEY_TO_SHAPE_ID_MAP,
    MAX_SNAP_DISTANCE_M,
)
from src._data_loaders import (
    get_selected_shapes,
    get_traffic_signals,
    list_available_service_dates,
)

from src.match_shapes_vp import project_points_on_shape

In [ ]:
# for caltrans network only

import os

#os.environ["REQUESTS_CA_BUNDLE"] = r"C:\Users\s163107\Documents\CTROOTCA01.cer"

In [ ]:
SHAPE_KEY = "105"
SHAPE_ID = SHAPE_KEY_TO_SHAPE_ID_MAP[SHAPE_KEY]
SEGMENT_START_SIGNAL = 29
SEGMENT_END_SIGNAL = SEGMENT_START_SIGNAL + 1
SEGMENT_BUFFER_LENGTH = 50
SPEED_BUCKET_MPH = 4
METERS_PER_SECOND_TO_MPH = 2.23694

In [ ]:
from src._data_loaders import get_smoothed_vehicle_positions


service_dates = list_available_service_dates()
smoothed_vehicle_positions = get_smoothed_vehicle_positions(service_date=SERVICE_DATE, shape_id=SHAPE_ID)

shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, [SHAPE_ID])
signals = get_traffic_signals()
stops = gpd.read_file(f"data/stops_{SHAPE_ID}.geojson").to_crs(CA_NAD83_Albers)

In [ ]:
shape = shapes.iloc[0]  # TODO: everything should assume a single shape here

# Distance along the full shape where each bounding signal sits
signal_distances = project_points_on_shape(signals.geometry, shapes.geometry, MAX_SNAP_DISTANCE_M)
start_signal_distance = signal_distances.loc[SEGMENT_START_SIGNAL]
end_signal_distance = signal_distances.loc[SEGMENT_END_SIGNAL]

# Segment runs between the two signals, with a buffer on each side
start_segment_distance = start_signal_distance - SEGMENT_BUFFER_LENGTH
end_segment_distance = end_signal_distance + SEGMENT_BUFFER_LENGTH
segment_length = end_signal_distance - start_signal_distance

In [ ]:
# Limit the smoothed trajectories to this segment and re-reference distances so
# that 0 = start signal (negative values fall in the upstream buffer).
in_segment = smoothed_vehicle_positions["distance_along_shape_smoothed"].between(
    start_segment_distance, end_segment_distance
)
smoothed_in_segment = smoothed_vehicle_positions.loc[in_segment].assign(
    distance_along_segment=lambda smoothed: smoothed["distance_along_shape_smoothed"] - start_signal_distance
)

In [ ]:
# Reference the stops to the same coordinate system (0 = start signal)
stop_distances = project_points_on_shape(stops, shapes.geometry, MAX_SNAP_DISTANCE_M) - start_signal_distance
stops_in_segment = stop_distances[
    (stop_distances >= -SEGMENT_BUFFER_LENGTH) & (stop_distances <= segment_length + SEGMENT_BUFFER_LENGTH)
]
print(f"start signal at 0 m, end signal at {segment_length:.0f} m")
print(f"{len(stops_in_segment)} stop(s) in segment at: {stops_in_segment.round(1).to_list()} m")

# Bucket each smoothed position's speed into SPEED_BUCKET_MPH-wide bands
speed_mph = smoothed_in_segment["speed_m_per_s"] * METERS_PER_SECOND_TO_MPH
speed_bucket_edges = np.arange(
    np.floor(speed_mph.min() / SPEED_BUCKET_MPH) * SPEED_BUCKET_MPH,
    np.ceil(speed_mph.max() / SPEED_BUCKET_MPH) * SPEED_BUCKET_MPH + SPEED_BUCKET_MPH,
    SPEED_BUCKET_MPH,
)
speed_bucket = pd.cut(speed_mph, bins=speed_bucket_edges, right=False)

# Stacked histogram: distance along segment (x), position count (y), speed band (color).
# Slow bands are red and fast bands green so congestion stands out.
distances_per_speed_bucket = [
    smoothed_in_segment.loc[speed_bucket == bucket, "distance_along_segment"]
    for bucket in speed_bucket.cat.categories
]
speed_bucket_labels = [
    f"{interval.left:.0f}–{interval.right:.0f} mph" for interval in speed_bucket.cat.categories
]
speed_bucket_colors = list(plt.cm.RdYlGn(np.linspace(0, 1, len(speed_bucket.cat.categories))))

# Positions with no computed speed get their own band so they aren't dropped
unknown_speed_distances = smoothed_in_segment.loc[speed_mph.isna(), "distance_along_segment"]
if not unknown_speed_distances.empty:
    distances_per_speed_bucket.append(unknown_speed_distances)
    speed_bucket_labels.append("unknown")
    speed_bucket_colors.append("lightgray")

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(
    distances_per_speed_bucket,
    bins=30,
    stacked=True,
    color=speed_bucket_colors,
    label=speed_bucket_labels,
)
ax.axvline(0, color="black", linestyle="--", label="start signal (segment start)")
ax.axvline(segment_length, color="black", linestyle="-.", label="end signal (segment end)")
for i, stop_distance in enumerate(stops_in_segment):
    ax.axvline(stop_distance, color="gray", linestyle=":", label="stop" if i == 0 else None)
ax.set_xlabel("distance along segment (m), 0 = start signal")
ax.set_ylabel("number of positions")
ax.legend(title="speed band", bbox_to_anchor=(1.02, 1), loc="upper left")

In [ ]:
smoothed_in_segment

In [ ]:
# Prepare data for the segment maps
import folium

from src._data_loaders import get_matched_vehicle_positions

FEET_TO_METERS = 0.3048
PING_BUFFER_FEET = 100
TRANSIT_SPEED_COLUMNS = [
    "TRANSIT_MIN_SPEED",
    "TRANSIT_MAX_SPEED",
    "TRANSIT_AVG_SPEED",
    "TWS_VEHICLE_SPEED",
]

segment_geometry = shapely.ops.substring(shape.geometry, start_segment_distance, end_segment_distance)
segment = gpd.GeoSeries([segment_geometry], crs=shapes.crs)
segment_buffer = segment_geometry.buffer(PING_BUFFER_FEET * FEET_TO_METERS)

# Attach the smoothing function's speed to each ping: for each trip, take the smoothed
# position nearest in space (distance along the shape) to the ping. TRIP_KEY is cast to a
# common string dtype on both sides so merge_asof can group on it.
matched_pings = get_matched_vehicle_positions(service_date=SERVICE_DATE, shape_id=SHAPE_ID)
pings_with_smoothed_speed = pd.merge_asof(
    matched_pings.astype({"TRIP_KEY": "string"}).sort_values("distance_along_shape").dropna(subset=["distance_along_shape"]),
    smoothed_vehicle_positions.astype({"TRIP_KEY": "string"}).sort_values(
        "distance_along_shape_smoothed"
    )[["TRIP_KEY", "distance_along_shape_smoothed", "speed_m_per_s"]],
    left_on="distance_along_shape",
    right_on="distance_along_shape_smoothed",
    by="TRIP_KEY",
    direction="nearest",
)
pings_with_smoothed_speed["smoothed_speed_mph"] = (
    pings_with_smoothed_speed["speed_m_per_s"] * METERS_PER_SECOND_TO_MPH
)
pings_with_smoothed_speed = gpd.GeoDataFrame(
    pings_with_smoothed_speed, geometry="geometry", crs=matched_pings.crs
)

# Pings within a 100 ft buffer of the segment, keeping the smoothed speed alongside the
# reported transit speeds so they can be compared in the tooltip.

pings_near_segment = pings_with_smoothed_speed.loc[
    pings_with_smoothed_speed.intersects(segment_buffer)
]

# Stops within the same buffer, drawn as star markers
stops_near_segment = stops.loc[stops.intersects(segment_buffer), ["geometry"]]

default_display_columns = ["EVENT_TYPE", "smoothed_speed_mph", *TRANSIT_SPEED_COLUMNS]
def build_segment_map(pings, display_columns=default_display_columns):
    """Map the segment, the given pings (colored by EVENT_TYPE), and stops as star markers."""
    segment_map = segment.explore()
    pings[[*display_columns, pings.geometry.name]].explore(
        m=segment_map, column="EVENT_TYPE", categorical=True, legend=True, marker_kwds={"radius": 3}
    )
    stops_near_segment.explore(
        m=segment_map,
        marker_type="marker",
        marker_kwds={"icon": folium.Icon(icon="star", prefix="fa", color="green")},
    )
    return segment_map

In [ ]:
# Speed comparisons: all pings near the segment, with smoothed vs. reported speeds in the tooltip
build_segment_map(pings_near_segment.loc[~pings_near_segment["EVENT_TYPE"].isin([3,4])])

In [ ]:
POTENTIAL_DWELL_EVENT_COLUMNS = [
    "STOP_BACK_DOOR_ENTRY",
    "STOP_FRONT_DOOR_ENTRY",
    "STOP_FRONT_DOOR_EXIT",
    "STOP_BACK_DOOR_EXIT",
    "STOP_REAR_DOOR_ONLY_ENTRY",
    "STOP_REAR_DOOR_ONLY_EXIT",
    "STOP_CENTER_DOOR_ONLY_ENTRY",
    "STOP_CENTER_DOOR_ONLY_EXIT",
    "STOP_AUX2_DOOR_ENTRY",
    "STOP_AUX2_DOOR_EXIT",
    "TOT_DWELL_TIME",
    "TIMEPOINT_STATUS"
]

# Door columns: only pings whose EVENT_TYPE is a door open/close event
door_pings_near_segment = pings_near_segment.loc[
    pings_near_segment[POTENTIAL_DWELL_EVENT_COLUMNS].notna().any(axis=1)
]
build_segment_map(door_pings_near_segment, display_columns=[*default_display_columns, *POTENTIAL_DWELL_EVENT_COLUMNS])

Dwell findings:
- door columns seem to be all 0 when EVENT_TYPE = 4
- door columns are nonzero when EVENT_TYPE = 3. This event is always triggered arund around the stop point
- door columns are nonzero when EVENT_TYPE = 5. However, this event is always triggered before the signal
- dwell time is < 10 s when EVENT_TYPE is 4
- dwell time is > 10 s when EVENT_TYPE is 3
- dwell time = 0 s when EVENT_TYPE = 5
- EVENT_TYPE = 16 seems to occur when the stop is a timepoint? However, this doesn't seem to correspong with EVENT_TYPE = 5 v EVENT_TYPE = 3.

Therefore
- if it is a EVENT_TYPE = 5 stop, there is no way to determine dwell time from clever data, but we can tell whether there was a stop event?
- otherwise, we can associate dwell time to an ID to calculate dwell

Thoughts for signal delay:
- Assume for now we know dwell time
- Find times for overnight trips
- Create copy of pings without events associated with dwells
- Get smoothed trips for pings with and pings without
- Using one without stop pings, determine if bus speed decreases before the stop
- Use this to determine whether vehicles stop at the intersection
- Get segment time difference between vehicles that stop, and vehicles that don't stop at intersection
- Use this + probability of stopping to calculate signal delay

Other thought: it looks like we might be able to get higher quality data from other agencies (ac transit, sbmtd, bbb, samtrans). do we want to continue trying to develop interseciton delay with messy culver data, or should we focus on outreach and other metrics?